# DS2002 · APIs 101 and Ingestion with Retries

**Studio — 2026-10-07 · Fall 2026**  
**Class time:** 45 minutes

---

## Data that lives on somebody else's computer

Reading days took Monday, so this is the only meeting this week and we are covering both halves of working with an API: getting the data, and staying upright when the request fails.

That second half is not optional. Every pipeline you have written so far reads a file that is either there or not. An API is different — it can be slow, it can rate-limit you, it can return a 500 for ninety seconds and then work fine. Your midterm depends on a weather API, so today you build the version that survives that.

We use **Open-Meteo**: real historical weather, free, no key required.

### What a request actually consists of

```
https://archive-api.open-meteo.com/v1/archive ? latitude=38.03 & start_date=2026-10-10 & ...
|------------------ base URL -------------------|  |--------------- query parameters ---------------|
```

You send that; the server sends back a **status code** and a **body**. The status code is the first thing to look at, every time.

| Code | Meaning | What you should do |
|---|---|---|
| 200 | Fine | Parse the body |
| 400 | Your request was malformed | Fix your parameters. Retrying will not help |
| 404 | Wrong URL | Fix the URL. Retrying will not help |
| 429 | You are asking too fast | Slow down and retry |
| 500–504 | Their server had a problem | Wait and retry |

That last column is the whole design of the retry logic you will write in a few minutes: retry the server's problems, never retry your own mistakes.

### Build 1 — see the URL before you send it

`requests` builds the query string from a dict for you. Print it once so the mapping between your dict and the actual URL stops being abstract.

In [13]:
import requests, pandas as pd

BASE = 'https://archive-api.open-meteo.com/v1/archive'
params = {
    'latitude': 38.03, 'longitude': -78.51,
    'start_date': '2026-09-01', 'end_date': '2026-09-03',
    'hourly': 'temperature_2m,precipitation',
    'timezone': 'America/New_York',
}

resp = requests.get(BASE, params=params, timeout=15)
print('status:', resp.status_code)
print(resp.text[:400])

status: 200
{"latitude":37.996483,"longitude":-78.50809,"generationtime_ms":0.5539655685424805,"utc_offset_seconds":-14400,"timezone":"America/New_York","timezone_abbreviation":"GMT-4","elevation":176.0,"hourly_units":{"time":"iso8601","temperature_2m":"°C","precipitation":"mm"},"hourly":{"time":["2026-09-01T00:00","2026-09-01T01:00","2026-09-01T02:00","2026-09-01T03:00","2026-09-01T04:00","2026-09-01T05:00",


Now send it. Note the three things this cell does that a careless version would skip: it sets a **timeout**, it looks at the **status code**, and it prints the top-level shape of the response instead of assuming it knows what came back.

In [14]:
resp = requests.get(BASE, params=params, timeout=15)
print('status:', resp.status_code)

if resp.status_code != 200:
    print('request failed:', resp.text[:300])
else:
    print('keys:', list(resp.json().keys()))
    print('units:', resp.json()['hourly_units'])

status: 200
keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
units: {'time': 'iso8601', 'temperature_2m': '°C', 'precipitation': 'mm'}


Open-Meteo hands back parallel lists — one list of timestamps, one list per variable. That shape drops straight into a DataFrame.

In [15]:
hourly = pd.DataFrame(resp.json()['hourly'])
hourly['time'] = pd.to_datetime(hourly['time'])
print(hourly.shape)
hourly.head()

(72, 3)


,time,temperature_2m,precipitation
0,2026-09-01 00:00:00,24.1,0.0
1,2026-09-01 01:00:00,23.6,0.0
2,2026-09-01 02:00:00,22.0,0.0
3,2026-09-01 03:00:00,22.2,0.0
4,2026-09-01 04:00:00,22.2,0.0


### Build 2 — what failure actually looks like

Send a request that is wrong on purpose: an end date before the start date. Read the status code and the body. The body usually tells you exactly what you got wrong, and most people never look at it.

In [16]:
bad = dict(params, start_date='2026-10-11', end_date='2026-10-09')
r = requests.get(BASE, params=bad, timeout=15)

print('status:', r.status_code)
print('body:', r.text[:300])

status: 400
body: {"reason":"Bad Request","error":true}


Now the trap. This is the most damaging pattern in data engineering, and it looks responsible:

```python
try:
    wx = fetch_weather(date)
except Exception:
    wx = pd.DataFrame()   # <- never do this
```

An empty frame from a failed call and an empty frame from a genuinely dry day are indistinguishable downstream. Your join silently drops rows, your chart shows no rain, and you present a conclusion built on a network error.

A failure must either raise, or be recorded as a failure. It must never disguise itself as data.

**TODO:** write `fetch_json` so it returns parsed JSON on success and raises a *useful* error otherwise — one that includes the status code and the start of the body.

In [17]:
def fetch_json(url, params, timeout=15):
    """Return parsed JSON, or raise an error that says what went wrong."""
    r = requests.get(url, params=params, timeout=timeout)
    if r.status_code != 200:
        raise RuntimeError(f'HTTP {r.status_code}: {r.text[:200]}')
    return r.json()

# Should work:
print(list(fetch_json(BASE, params).keys()))

# Should raise something informative:
try:
    fetch_json(BASE, bad)
except Exception as e:
    print('raised:', e)

['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
raised: HTTP 400: {"error":true,"reason":"Bad Request"}


### Build 3 — retry the failures worth retrying

A 500 or a timeout usually means "try again in a moment." A 400 means "you are wrong, and you will still be wrong in a moment."

Back off between attempts instead of hammering: wait 1 second, then 2, then 4. If you retry instantly in a loop you look like an attack and get rate-limited.

First, prove the logic without the network. `flaky()` fails twice and then succeeds, so the test is deterministic.

**TODO:** implement `with_retries`.

In [19]:
import time

attempts = {'n': 0}

def flaky():
    """Fails the first two times it is called, then succeeds."""
    attempts['n'] += 1
    if attempts['n'] < 3:
        raise TimeoutError(f"simulated timeout (call {attempts['n']})")
    return {'ok': True, 'calls_needed': attempts['n']}

def with_retries(fn, tries=4, base_delay=0.1):
    """Call fn(), retrying on failure with an increasing delay.
    Re-raise if every attempt fails."""
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            if attempt == tries - 1:
                raise                                  # last attempt: let it escape
            print(f'attempt {attempt + 1} failed: {e}')
            time.sleep(base_delay * (2 ** attempt))

attempts['n'] = 0     # reset so re-running this cell gives the same result
print(with_retries(flaky))

attempt 1 failed: simulated timeout (call 1)
attempt 2 failed: simulated timeout (call 2)
{'ok': True, 'calls_needed': 3}


**TODO:** now wire the same idea into a real fetch. Retry on 429 and 5xx and on network errors. Do **not** retry on 400 or 404 — raise immediately, because retrying a bad request just wastes everyone's time.

In [20]:
RETRY_STATUSES = {429, 500, 502, 503, 504}

def fetch_with_retries(url, params, tries=4, base_delay=0.5):
    last = None
    for attempt in range(tries):
        try:
            r = requests.get(url, params=params, timeout=15)
        except requests.RequestException as e:
            last = e                                    # network error: retry
        else:
            if r.status_code == 200:
                return r.json()
            if r.status_code not in RETRY_STATUSES:     # our mistake: raise now
                raise RuntimeError(f'HTTP {r.status_code}: {r.text[:200]}')
            last = f'HTTP {r.status_code}'              # their problem: retry
        if attempt < tries - 1:
            time.sleep(base_delay * (2 ** attempt))
    raise RuntimeError(f'gave up after {tries} attempts: {last}')

data = fetch_with_retries(BASE, params)
print('got', len(data['hourly']['time']), 'hourly readings')

got 72 hourly readings


### Build 4 — stop asking the same question twice

Re-running a notebook cell should not mean re-hitting somebody's server. A small cache makes your notebook faster, makes your results stable while you work, and keeps you off the rate limit.

**TODO:** finish `cached_fetch`.

In [21]:
_cache = {}

def cache_key(params):
    return tuple(sorted(params.items()))

def cached_fetch(url, params):
    key = cache_key(params)
    if key in _cache:
        print('cache hit')
        return _cache[key]
    result = fetch_with_retries(url, params)
    _cache[key] = result
    return result

cached_fetch(BASE, params)
cached_fetch(BASE, params)   # this one should not touch the network
print('cached requests:', len(_cache))

cache hit
cached requests: 1


### Build 5 — hourly readings into the shape you can join

Sales data is daily. Weather came back hourly. Somebody has to reshape, and the choice of aggregation is a real decision: rain **sums** over a day, temperature **maxes**.

**TODO:** produce one row per calendar day with total precipitation and max temperature.

In [22]:
hourly['date'] = hourly['time'].dt.date

daily = (
    hourly.groupby('date')
    .agg(precipitation_total=('precipitation', 'sum'),
         temp_max=('temperature_2m', 'max'))
    .reset_index()
)
daily

,date,precipitation_total,temp_max
0,2026-09-01,0.1,34.9
1,2026-09-02,0.7,36.0
2,2026-09-03,0.7,33.0


**TODO:** which single hour across the range had the most precipitation? Print the timestamp and the amount.

In [23]:
units = resp.json()['hourly_units']['precipitation']
wettest = hourly.loc[hourly['precipitation'].idxmax()]
print('wettest hour:', wettest['time'], '|', wettest['precipitation'], units)

wettest hour: 2026-09-03 17:00:00 | 0.4 mm


### Cite what you pulled

Every project in this course that uses an API has to say where the data came from. Not as a formality — six weeks from now you will need to know whether a number came from the archive endpoint or the forecast endpoint, and which coordinates you used.

In [24]:
from datetime import datetime

print('Source:  Open-Meteo historical archive API')
print('Endpoint:', BASE)
print('Params: ', params)
print('Pulled: ', datetime.now().strftime('%Y-%m-%d %H:%M'))

Source:  Open-Meteo historical archive API
Endpoint: https://archive-api.open-meteo.com/v1/archive
Params:  {'latitude': 38.03, 'longitude': -78.51, 'start_date': '2026-09-01', 'end_date': '2026-09-03', 'hourly': 'temperature_2m,precipitation', 'timezone': 'America/New_York'}
Pulled:  2026-10-06 22:41


---

## Checkpoint (participation)

Report the daily precipitation total your reshape produced, and what happened when you sent the deliberately bad request.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [27]:
# Checkpoint
daily_rows = len(daily)
wettest_day = daily.loc[daily['precipitation_total'].idxmax(), 'date']
bad_request_status = requests.get(BASE, params=bad, timeout=15).status_code
retry_rule = '400 (or 404): the request itself is wrong, so retrying sends the same bad request and gets the same error.'

print('daily rows:', daily_rows)
print('wettest day:', wettest_day)
print('bad request returned:', bad_request_status)
print('would not retry:', retry_rule)

daily rows: 3
wettest day: 2026-09-03
bad request returned: 400
would not retry: 400 (or 404): the request itself is wrong, so retrying sends the same bad request and gets the same error.
